# Let's build the GPT Tokenizer

跟随 Karpathy Lecture 8：从字符级走向 BPE（Byte Pair Encoding）。

- 视频：[Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE)
- 官方 Colab：[Open in Colab](https://colab.research.google.com/drive/1y0KnCFZvGVf_odSfcNAws6kcDD7HsI0L?usp=sharing)
- 参考实现：[karpathy/minbpe](https://github.com/karpathy/minbpe)
- 上一课笔记：Vault `nanoGPT · Let's build GPT`

上一课用的是**字符级** tokenizer（约 65 个字符）。
这一课要搞清：真正 GPT 为什么改用 **子词 token**，以及 BPE 怎么训练 / encode / decode。

## 本课目标

1. 为什么字符级不够，词级也不够
2. Unicode / UTF-8 / byte 和 token 的关系
3. BPE：数 pair 频率 → 合并 → 得到词表
4. `encode` / `decode`
5. 特殊 token、正则切分（GPT-2 风格）带来的坑


In [ ]:
print('ready for tokenizer lecture')

## 0. 从上一课接过来

字符级：

```text
"hello" → ['h','e','l','l','o'] → 5 个 token
```

优点：词表极小、实现简单。
缺点：序列很长，`block_size` 很快被占满；模型要自己拼单词。

目标：找到折中——**子词（subword）**。


## 1. 先用手搓一次最小 BPE（预习）

下面用极短文本演示算法骨架；跟视频时再对照扩展到完整实现。


In [ ]:
text = 'aaabdaaabac'
print('raw:', text)
print('chars:', list(text))

# 一开始每个字符自己就是一个 token
tokens = list(text)
print('tokens:', tokens)


def get_stats(ids):
    """统计相邻 pair 出现次数"""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


stats = get_stats(tokens)
best = max(stats, key=stats.get)
print('most common pair:', best, 'count=', stats[best])
print('top pairs:', sorted(stats.items(), key=lambda kv: -kv[1])[:5])

In [ ]:
def merge(ids, pair, new_id):
    """把序列里所有相邻 pair 合并成 new_id"""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# 演示合并一次：最常见 pair ('a','a') → 新符号 'Z'
tokens2 = merge(tokens, ('a', 'a'), 'Z')
print('after one merge:', tokens2)
print('length:', len(tokens), '→', len(tokens2))

## 2. 训练循环直觉

```text
重复 N 次:
  1. 统计当前序列里所有相邻 pair 的频率
  2. 选出最高频 pair
  3. 合并成一个新 token，写入词表
  4. 用新 token 重写整段序列
```

词表越大 / 合并轮数越多 → 平均每个 token 覆盖越长的文本片段 → 同样 `block_size` 能装更多语义。


## 跟视频时建议关注

- UTF-8：字符串 ↔ bytes ↔ token id
- 为什么很多实现从 256 个 byte token 起步
- regex 预切分（GPT-2）如何改变 merge 候选
- 特殊 token：`<|endoftext|>` 等
- tokenization 会导致哪些模型“怪行为”

看完一节就在下面追加你的疑问与小结。
